# **Implementation**
## **The goal of the notebook**
---
---
The goal of this example is to provide an example for using the BOSS method to find the preferred adsoprtion site of OH intermediate on a PtN4-carbon single-atom catalyst.

**Workflow**

1.   Creating the surface + adsorbate
2.   Finding the preferred asdorption site for OH
---
---

---
---
## **Installation of packages**

---

First thing first, we need to import the neccessary functions.

---
---

In [ ]:
##############
##-Imports-###
##############

##-Imports for ASE
from ase import Atoms, Atom
from ase.build import add_adsorbate, fcc111, graphene, molecule
from ase.cluster.cubic import FaceCenteredCubic
from ase.data.pubchem import pubchem_atoms_search
from ase.optimize import GPMin
from ase.io import read, write
from ase.visualize import view
from ase.visualize.plot import plot_atoms

##-Imports for Boss
from boss.bo.bo_main import BOMain

##-Import for TBLite
from tblite.ase import TBLite

##-General imports
from copy import deepcopy as dc
import matplotlib.pyplot as plt
import numpy as np

---
---
## **Creating the surface**

---
### **1.   Creating the surface + adsorbate**
2.   Finding the preferred asdorption site for OH
---
---

We create the surface using ASE functions and run single point calculations on H2O, H2 and OH.

---
---

In [ ]:
##########################
##-Creating the surface-##
##########################

PtN4 = graphene(size=(4,4,1))
for i in (0, 3, 8, 11):
    PtN4[i].symbol = "N"
PtN4.append(Atom('Pt', position=(PtN4[1].position+PtN4[10].position)/2))
del PtN4[[10, 1]]
PtN4.cell[1,0] *= -1
PtN4.cell[2][2] = 6
shift = PtN4.cell[:2,:2].sum(axis=0) / 2 - PtN4[-1].position[:2]
PtN4.translate([shift[0], shift[1], 0.0])
view(PtN4*(1,1,1), start_rotation=False)

In [ ]:
######################
##-Adding adsorbate-##
######################

##-This sets up our adsorbate with the H on top for rotational reference
OH = Atoms('OH', positions=[[0,0,0],
                                  [0,0,0.96]])

water = Atoms('H2O',positions=[[0.75,0.58,0],[-0.75,0.58,0],[0,0,0]])
hydrogen = Atoms('H2', positions=[[0,0,0],[0.7,0,0]])

In [ ]:
###############################
##-Single point calculations-##
###############################

PtN4.calc = TBLite(method="GFN1-xTB")

water.calc = TBLite(method="GFN1-xTB")
hydrogen.calc = TBLite(method="GFN1-xTB")

PtN4.get_potential_energy()
water.get_potential_energy()
hydrogen.get_potential_energy()

---
---
## **Preferred adsorption site**

---
1.   Creating the surface + adsorbate
### **2.   Finding the preferred asdorption site for OH**
---
---

In this section, we define two functions. The first function places our adsorbate on 6 coordinates. The second function represents BOSS taking the coordinates and returning a single scalar. Now using the defined functions BOSS object and run the calculations. From the calculations we get the predicted global energy minimum.

---
---

In [ ]:
########################
##-Defining functions-##
########################

##-This function will place our adsorbate on the function based on 6 coordinates
def build_system(x,y,z,rx,ry,rz, slab: Atoms, adsorbate: Atoms,) -> Atoms:
    slab = slab.copy()
    adsorbate = adsorbate.copy()
    adsorbate.rotate(rx,'x')
    adsorbate.rotate(ry,'y')
    adsorbate.rotate(rz,'z')
    adsorbate.translate((x,y,z))

    full_system = slab + adsorbate
    return full_system

##-This function represents one step of boss taking the 6 dimensional coordinates and returns a single scalar
def calculate_system(x,y,z,rx,ry,rz, slab: Atoms, adsorbate: Atoms, slab_en:float, adsorbate_en: float | list[float], calculator) -> float:
    full_system = build_system(x,y,z,rx,ry,rz, slab, adsorbate)
    full_system.calc = dc(calculator)
    full_system.calc.txt = f'full_system_{x}-{y}-{z}_{rx}-{ry}-{rz}.txt'

    if isinstance(adsorbate_en, list) or isinstance(adsorbate_en, tuple): adsorbate_en = sum(adsorbate_en)

    return full_system.get_potential_energy() - slab_en - adsorbate_en


In [ ]:
##########################
##-Optimizing structure-##
##########################

BO_obj = BOMain(
    f= lambda args: calculate_system(args[0,0],args[0,1],args[0,2],args[0,3],args[0,4],args[0,5],
                                     PtN4, OH,
                                     slab_en=PtN4.get_potential_energy(),
                                     adsorbate_en=(water.get_potential_energy(),
                                                   -0.5*hydrogen.get_potential_energy()),
                                     calculator=TBLite(method="GFN1-xTB", verbosity = 0)),
    bounds=np.array([[0,PtN4.cell[0,0] + PtN4.cell[1,0]],
                    [0,PtN4.cell[1,1]],
                    [0.5,PtN4.cell[2,2]],
                    [-90, 90],
                    [-90, 90],
                    [0, 360],]
                    ),
    kernel='rbf',
    initpts=5,
    iterpts=300
)

res = BO_obj.run()

print('Predicted global min: ', res.select('mu_glmin', -1))
print(res.select('x_glmin', -1))
final_structure = build_system(*res.select('x_glmin', -1), slab = PtN4, adsorbate = OH)
view(final_structure, viewer='x3d')

write('example_4_boss_final.xyz', final_structure)